# Autoresearch Experiment Analysis

Analysis of autonomous hyperparameter tuning results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated) with the canonical schema.
# Required column order:
# timestamp, commit, val_bpb, memory_gb, status, description
# timestamp is ISO-8601 (e.g. 2026-05-21T19:59:30-07:00).
df = pd.read_csv("results.tsv", sep="\t")
expected_cols = ["timestamp", "commit", "val_bpb", "memory_gb", "status", "description"]
if list(df.columns) != expected_cols:
    raise ValueError(f"results.tsv schema mismatch. Expected {expected_cols}, got {list(df.columns)}")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="raise", utc=True)
df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].astype(str).str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Time range (UTC): {df['timestamp'].min()} -> {df['timestamp'].max()}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    bpb = row["val_bpb"]
    desc = row["description"]
    ts = row["timestamp"]
    ts_text = ts.isoformat() if pd.notna(ts) else "n/a"
    print(f"  #{i:3d}  bpb={bpb:.6f}  mem={row['memory_gb']:.1f}GB  ts={ts_text}  {desc}")

## Val BPB Over Time

Track how the best (kept) val_bpb evolves as experiments progress. The running minimum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes and keep rows with valid timestamps for time-based plotting
valid = df[(df["status"] != "CRASH") & (df["timestamp"].notna())].copy()
valid = valid.sort_values("timestamp").reset_index(drop=True)
valid["exp_num"] = valid.index + 1

baseline_bpb = valid.loc[0, "val_bpb"]

# Y-axis tight around the interesting region (best kept to just above baseline)
kept_mask = valid["status"] == "KEEP"
kept_t = valid.loc[kept_mask, "timestamp"]
kept_bpb = valid.loc[kept_mask, "val_bpb"]
running_min = kept_bpb.cummin()

best = float(kept_bpb.min()) if len(kept_bpb) else float(baseline_bpb)
y_margin = max((baseline_bpb - best) * 0.25, 0.002)
y_min = best - y_margin
y_max = baseline_bpb + y_margin * 2

# Partition into in-range and out-of-range
in_range = valid[(valid["val_bpb"] >= y_min) & (valid["val_bpb"] <= y_max)]
out_of_range = valid[(valid["val_bpb"] < y_min) | (valid["val_bpb"] > y_max)]

# Plot in-range discarded as faint dots
disc = in_range[in_range["status"] == "DISCARD"]
ax.scatter(disc["timestamp"], disc["val_bpb"],
           c="#cccccc", s=220, alpha=0.55, zorder=2,
           label="Discarded", edgecolors="black", linewidths=0.5)

# Plot in-range kept as prominent green dots
kept_v = in_range[in_range["status"] == "KEEP"]
ax.scatter(kept_v["timestamp"], kept_v["val_bpb"],
           c="#2ecc71", s=320, zorder=4,
           label="Kept", edgecolors="black", linewidths=0.7)

# Experiment number inside each in-range dot
for _, row in in_range.iterrows():
    text_color = "black" if row["status"] == "DISCARD" else "white"
    ax.text(row["timestamp"], row["val_bpb"], f"{int(row['exp_num'])}",
            ha="center", va="center", fontsize=8,
            color=text_color, fontweight="bold", zorder=5)

# Out-of-range: draw a clipped triangle at the axis boundary with actual value
for _, row in out_of_range.iterrows():
    is_above = row["val_bpb"] > y_max
    clip_y = y_max if is_above else y_min
    marker = "^" if is_above else "v"
    color = "#2ecc71" if row["status"] == "KEEP" else "#aaaaaa"
    # Triangle marker at the boundary
    ax.scatter(row["timestamp"], clip_y, marker=marker,
               c=color, s=260, zorder=6, edgecolors="black", linewidths=0.8,
               clip_on=False)
    # Exp number inside triangle (offset slightly inward)
    offset = -0.0008 if is_above else 0.0008
    ax.text(row["timestamp"], clip_y + offset, f"{int(row['exp_num'])}",
            ha="center", va="center", fontsize=7,
            color="black" if row["status"] == "DISCARD" else "white",
            fontweight="bold", zorder=7)
    # Actual value annotation
    xytext_y = -18 if is_above else 12
    ax.annotate(f"{row['val_bpb']:.4f}",
                (row["timestamp"], clip_y),
                textcoords="offset points", xytext=(4, xytext_y),
                fontsize=7.5, color="#cc4400", ha="left", va="top" if is_above else "bottom",
                zorder=7)

# Running minimum step line
ax.step(kept_t, running_min, where="post",
        color="#27ae60", linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept in-range experiment with its description
for _, row in in_range[in_range["status"] == "KEEP"].iterrows():
    desc = str(row["description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (row["timestamp"], row["val_bpb"]),
                textcoords="offset points", xytext=(8, 8),
                fontsize=8.0, color="#1a7a3a", alpha=0.9,
                rotation=25, ha="left", va="bottom")

# Add triangle to legend
import matplotlib.lines as mlines
outlier_handle = mlines.Line2D([], [], marker="^", color="w", markerfacecolor="#aaaaaa",
                               markeredgecolor="black", markersize=9, label="Outlier (clipped)")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles + [outlier_handle], loc="upper right", fontsize=9)

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Time (UTC)", fontsize=12)
ax.set_ylabel("Validation BPB (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.grid(True, alpha=0.2)
ax.set_ylim(y_min, y_max)

fig.autofmt_xdate(rotation=25)
plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")


## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_bpb = df.iloc[0]["val_bpb"]
best_bpb = kept["val_bpb"].min()
best_row = kept.loc[kept["val_bpb"].idxmin()]

print(f"Baseline val_bpb:  {baseline_bpb:.6f}")
print(f"Best val_bpb:      {best_bpb:.6f}")
print(f"Total improvement: {baseline_bpb - best_bpb:.6f} ({(baseline_bpb - best_bpb) / baseline_bpb * 100:.2f}%)")
print(f"Best experiment:   {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: bpb={row['val_bpb']:.6f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's bpb
# (since experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_bpb"] = kept["val_bpb"].shift(1)
kept["delta"] = kept["prev_bpb"] - kept["val_bpb"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'BPB':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_bpb']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")